# Sionna 0.19 — Differentiable Ray-Tracing Calibration

Optimises material EM parameters and TX orientation using differentiable RT (DrJit / TF GradientTape) to minimise NMSE between simulated and measured RSSI.

**Prerequisites — run main notebook first:**
1. CELL 3 (scene built → `scene.xml` exists)
2. CELL 6c (receivers extracted → `receiver_locations.csv` + `measurements_with_pathloss.csv`)
3. CELL 7 (TX placed → `transmitter_positions.csv`)
4. CELL 9d/9e (baseline RMSE recorded — compare before/after calibration)

**Calibration pipeline:**
```
CELL 0 → 0c → 1 → 2 → [3 preview] → 4 → 5 → 6 → 7 → 8
Setup  Config  Coords  Scene        TX+RX  Baseline  RefCh  MatCal  TXOri  PostCal
```

**Expected output:** 3–8 dB RMSE improvement vs uncalibrated ITU-R P.2040-2 parameters

**GPU:** Tesla V100-SXM2-16GB · `cuda_ad_rgb` · `FORCE_CPU_RT=False`

## CELL 0 · Environment Setup & GPU Configuration

Same GPU setup as main notebook — `set_memory_growth` only, no `VirtualDeviceConfiguration`.

**Run this first, then restart kernel if imports fail.**

In [ ]:
import os, sys

import sys, json, csv, time, warnings, glob, re

# ── CPU/GPU mode — must be set before TF and Mitsuba imports ─────────────────
FORCE_CPU_RT = False  # False = GPU (cuda_ad_rgb); True = CPU (llvm_ad_rgb)
import xml.etree.ElementTree as ET
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import pandas as pd
import matplotlib; matplotlib.rcParams.update({'font.size': 11, 'figure.dpi': 100})
import matplotlib.pyplot as plt
from scipy.spatial import KDTree
from scipy import stats
from scipy.stats import spearmanr
from scipy.constants import speed_of_light as C
from pyproj import Transformer
from datetime import datetime

try:
    import seaborn as sns
    sns.set_theme(style='whitegrid')
except ImportError:
    pass

import tensorflow as tf
# ── GPU strategy ──────────────────────────────────────────────────────────────
# Root cause of DLPack "GPU:0 unknown device" error:
#   Sionna/DrJit creates DLPack capsules referencing the physical GPU.
#   If TF can't see the GPU it crashes consuming those capsules.
#
# Permanent fix for FORCE_CPU_RT=True:
#   Hide GPU from CUDA/Mitsuba (CUDA_VISIBLE_DEVICES='') BEFORE mitsuba import.
#   Mitsuba then has no CUDA device → uses llvm_ad_rgb, creates CPU DLPack only.
#   TF keeps seeing the GPU for its own ops (no OOM, no DLPack conflict).
#
# For FORCE_CPU_RT=False (GPU ray tracing):
#   TF capped at 1 GB so Mitsuba/DrJit gets remaining VRAM.
# ──────────────────────────────────────────────────────────────────────────────
if FORCE_CPU_RT:
    os.environ['CUDA_VISIBLE_DEVICES'] = ''   # hide GPU from CUDA/Mitsuba
    print('CUDA    : GPU hidden from CUDA (FORCE_CPU_RT=True) — Mitsuba will use llvm_ad_rgb')

gpus = tf.config.list_physical_devices('GPU')
if gpus and not FORCE_CPU_RT:
    # IMPORTANT: use memory_growth only — VirtualDeviceConfiguration creates
    # a virtual GPU with a different device index than the physical GPU.
    # DrJit DLPack capsules reference physical GPU:0; if TF's virtual GPU:0
    # doesn't match, consuming those capsules crashes with 'unknown device'.
    try:
        tf.config.experimental.set_memory_growth(gpus[0], True)
        print(f'TF GPU  : {gpus[0].name} — memory growth enabled (DLPack-safe)')
    except RuntimeError as _e:
        print(f'TF GPU  : memory growth failed ({_e}) — already initialised')
elif gpus:
    print(f'TF GPU  : {gpus[0].name} visible to TF (CUDA hidden from Mitsuba)')
else:
    print('TF GPU  : not detected')
tf.get_logger().setLevel('ERROR')
tf.random.set_seed(42)

_HAS_MI = False
try:
    import mitsuba as mi
    if FORCE_CPU_RT:
        _MI_VARIANTS = ['llvm_ad_rgb', 'scalar_rgb']
    else:
        _MI_VARIANTS = ['cuda_ad_rgb', 'llvm_ad_rgb', 'scalar_rgb']
    for _var in _MI_VARIANTS:
        try:
            mi.set_variant(_var)
            _HAS_MI = True
            print(f'Mitsuba : {mi.variant()}  (FORCE_CPU_RT={FORCE_CPU_RT})')
            break
        except Exception:
            continue
    if not _HAS_MI:
        print(f'Mitsuba : imported but no usable variant found')
except ImportError:
    print('Mitsuba : NOT installed')

import sionna
from sionna.rt import load_scene, RadioMaterial, PlanarArray, Transmitter, Receiver, Camera

_HAS_OFDM = False
try:
    from sionna.channel import cir_to_ofdm_channel, subcarrier_frequencies
    _HAS_OFDM = True; print('OFDM    : OK  (sionna.channel)')
except (ImportError, AttributeError):
    try:
        from sionna.channel.ofdm import cir_to_ofdm_channel, subcarrier_frequencies
        _HAS_OFDM = True; print('OFDM    : OK  (sionna.channel.ofdm)')
    except: print('OFDM    : NOT found – power fallback will be used')

# ── Mitsuba – set variant BEFORE importing sionna ────────────────────────────
# Sionna 0.19 picks up whatever mi.set_variant() is active at import time.
# FORCE_CPU_RT=True → llvm_ad_rgb (CPU, no VRAM limit, works for full 81k-bld scene)
# FORCE_CPU_RT=False → cuda_ad_rgb (GPU, faster but OOMs on large BVH)
#
# IMPORTANT: this block must run before `import sionna` above.
# Change FORCE_CPU_RT here AND in CELL 0c — they must match.

_HAS_RIO = False
try:
    import rasterio as rio; _HAS_RIO = True; print('rasterio: OK')
except ImportError:
    print('rasterio: NOT available')

_HAS_OSM = False
try:
    import osmnx as ox
    from shapely.geometry import box, Polygon, MultiPolygon
    from shapely.ops import unary_union
    _HAS_OSM = True; print('osmnx   : OK')
except ImportError:
    print('osmnx   : NOT available – pip install osmnx shapely')

print(f'Python  : {sys.version.split()[0]}')
print(f'TF      : {tf.__version__}')
print(f'Sionna  : {sionna.__version__}')

def _safe(v):
    if hasattr(v, 'numpy'): return float(v.numpy())
    if hasattr(v, 'item'):  return float(v.item())
    return float(v)

def _to_numpy(t):
    if isinstance(t, tuple): return t[0].numpy() + 1j * t[1].numpy()
    if hasattr(t, 'numpy'): return t.numpy()
    return np.array(t)

def _cm_to_numpy(cm_obj):
    for attr in ('path_gain', 'rss', 'as_tensor'):
        if not hasattr(cm_obj, attr): continue
        val = getattr(cm_obj, attr)
        arr = val() if callable(val) else val
        if hasattr(arr, 'numpy'): return arr.numpy()
        try: return np.array(arr)
        except: pass
    raise AttributeError('Cannot extract path_gain from CoverageMap.')


## CELL 0c · Calibration Configuration

**All paths and RF parameters must match the main simulation notebook.**

| Parameter | Value | Must match main notebook? |
|---|---|---|
| `FREQUENCY_HZ` | 3602.5 MHz | ✅ Yes |
| `EIRP_DBM` | 54.0 dBm | ✅ Yes |
| `SYS_GAIN` | 16.0 dB | ✅ Yes |
| `NOISE_FLOOR` | −115 dBm | ✅ Yes |
| `MAX_DEPTH` | 15 | ✅ Yes |
| `MEASUREMENT_CSV` | auto from RX_CSV | ✅ Yes |
| `CALIB_STEPS` | 100 | Calibration only |
| `CALIB_LR` | 5e-3 | Calibration only |
| `CALIB_NUM_SAMP` | 500K | Calibration only |
| `CALIB_DEPTH` | 6 | Faster than 12 for grad steps |

In [ ]:
# ── Paths — must match main simulation notebook ───────────────────────────────
import os
CITY_NAME    = 'Nottingham'
SCENE_VERSION = '11km'  # match main notebook CELL 0c
_suffix       = '_11km' if SCENE_VERSION == '11km' else ''
BASE_DIR     = os.path.expanduser(f'~/Documents/FYP2026/{CITY_NAME.lower()}{_suffix}')
OUT_DIR      = os.path.join(BASE_DIR, 'results_diff_rt')
SCENE_XML    = os.path.join(BASE_DIR, 'scene', 'scene.xml')
TX_CSV       = os.path.join(BASE_DIR, 'transmitter_positions.csv')
RX_CSV       = os.path.join(BASE_DIR, 'receiver_locations.csv')
DEM_TIFF     = '/home/georgeskai/Documents/Region/nottingham3602/uk_terrain_nottingham_aoi.tif'
MEASUREMENT_CSV = os.path.join(os.path.dirname(RX_CSV), 'measurements_with_pathloss.csv')
os.makedirs(OUT_DIR, exist_ok=True)

# ── Scene GPS bounds (must match main notebook CELL 0c) ───────────────────────
if SCENE_VERSION == '11km':
    WEST, EAST   = -1.409134, -1.245094
    SOUTH, NORTH =  52.910276, 53.009090
else:
    WEST, EAST   = -1.447449, -1.206779
    SOUTH, NORTH =  52.918218, 53.001149
center_lon   = (WEST + EAST) / 2
center_lat   = (SOUTH + NORTH) / 2
UTM_EPSG     = 32630
BNG_EPSG     = 27700

# ── RF link budget (must match main notebook CELL 0c) ─────────────────────────
FREQUENCY_HZ   = 3602.5e6     # 3602.5 MHz (Ofcom) — match main notebook
EIRP_DBM       = 54.0
SYS_GAIN       = 16.0
NOISE_FLOOR    = -115.0
NOISE_FLOOR_DBM = NOISE_FLOOR
TX_LON, TX_LAT = -1.2559, 52.9863
TX_AGL_M, RX_AGL_M = 17.0, 1.5
MAX_DEPTH      = 15            # match main notebook
GRID_SIZE_M    = 5.0          # match main notebook

# ── Calibration hyper-parameters ──────────────────────────────────────────────
CALIB_STEPS    = 100           # gradient steps for material calibration
CALIB_LR       = 5e-3          # Adam learning rate
CALIB_NUM_SAMP = 500_000       # rays per gradient step (keep low for speed)
CALIB_DEPTH    = 6             # max_depth during calibration (faster than 12)

ORI_STEPS      = 50            # TX orientation optimisation steps
ORI_LR         = 1e-2

# ── Post-calibration coverage map ─────────────────────────────────────────────
NUM_SAMPLES_CM = 50_000_000    # 5 × 10M for post-cal map

print('Calibration config loaded — ' + CITY_NAME)
print('Scene XML       : ' + SCENE_XML)
print('Measurements    : ' + MEASUREMENT_CSV)
print('Output dir      : ' + OUT_DIR)
print('Frequency       : ' + str(FREQUENCY_HZ/1e6) + ' MHz')
print('CALIB_STEPS     : ' + str(CALIB_STEPS) + '  LR: ' + str(CALIB_LR))
print('CALIB_NUM_SAMP  : ' + str(CALIB_NUM_SAMP))


## CELL 1 · Coordinate Utilities + DEM Elevation

Defines `gps_to_local()`, `local_to_gps()`, `ray_cast_ground_z()` — identical to main notebook.

**Must run before any cell that uses GPS coordinates.**

In [ ]:
gps_to_utm = Transformer.from_crs('EPSG:4326', f'EPSG:{UTM_EPSG}', always_xy=True)
utm_to_gps = Transformer.from_crs(f'EPSG:{UTM_EPSG}', 'EPSG:4326', always_xy=True)
utm_to_bng = Transformer.from_crs(f'EPSG:{UTM_EPSG}', f'EPSG:{BNG_EPSG}', always_xy=True)

utm_center_x, utm_center_y = gps_to_utm.transform(center_lon, center_lat)
print(f'UTM center : ({utm_center_x:.1f}, {utm_center_y:.1f})')

def gps_to_local(lon, lat, height=0.0):
    ux, uy = gps_to_utm.transform(lon, lat)
    return float(ux - utm_center_x), float(uy - utm_center_y), float(height)

def local_to_gps(x, y):
    lon, lat = utm_to_gps.transform(_safe(x) + utm_center_x, _safe(y) + utm_center_y)
    return float(lon), float(lat)

dem_data = dem_nodata = dem_tf = dem_crs = None
if _HAS_RIO and os.path.exists(DEM_TIFF):
    _src      = rio.open(DEM_TIFF)
    dem_data  = _src.read(1).astype(np.float32)
    dem_nodata= _src.nodata
    dem_tf    = _src.transform
    dem_crs   = str(_src.crs)
    print(f'DEM     : {dem_data.shape}  nodata={dem_nodata}  CRS={dem_crs}')
else:
    print('DEM     : not loaded (rasterio missing or file absent)')

_is_bng_dem = dem_crs is not None and ('27700' in dem_crs or 'OSGB' in dem_crs.upper())

def get_dem_elevation(local_x, local_y):
    if dem_data is None: return 0.0
    utm_x = _safe(local_x) + utm_center_x
    utm_y = _safe(local_y) + utm_center_y
    px, py = (utm_to_bng.transform(utm_x, utm_y) if _is_bng_dem
              else utm_to_gps.transform(utm_x, utm_y))
    col_f, row_f = ~dem_tf * (px, py)
    r, c = int(np.floor(row_f)), int(np.floor(col_f))
    H, W = dem_data.shape
    if 0 <= r < H-1 and 0 <= c < W-1:
        dr, dc = row_f - r, col_f - c
        z = ((1-dr)*(1-dc)*dem_data[r,c]   + (1-dr)*dc*dem_data[r,c+1] +
              dr*(1-dc)*dem_data[r+1,c]    + dr*dc*dem_data[r+1,c+1])
        if dem_nodata is None or not np.isclose(float(z), dem_nodata):
            return float(z)
    return 0.0

def ray_cast_ground_z(x, y, max_height=2000.0):
    if _HAS_MI:
        try:
            ray = mi.Ray3f(mi.Point3f(float(x), float(y), max_height),
                           mi.Vector3f(0.0, 0.0, -1.0))
            si = scene.mi_scene.ray_intersect(ray)
            if si.is_valid():
                z_val = si.p.z
                return float(z_val.item()) if hasattr(z_val, 'item') else float(z_val)
        except Exception: pass
    return get_dem_elevation(x, y)

print('Coordinate utilities ready.')
print(f'  gps_to_local({center_lon:.4f}, {center_lat:.4f}) → {gps_to_local(center_lon, center_lat)[:2]}')


## CELL 2 · Load Scene & Assign Materials

Loads `scene.xml` and applies ITU-R P.2040-2 material parameters with LambertianPattern.

**Requires:** `scene.xml` generated by main notebook CELL 3

**Materials:** same `_MAT_PARAMS` as main notebook — concrete, brick, glass, metal, asphalt, vegetation, ground types

**LambertianPattern + xpd_coefficient** applied to all materials (same as main notebook)

In [ ]:
XML_OK = os.path.exists(SCENE_XML)
if not XML_OK:
    raise RuntimeError(
        f'scene.xml not found at {SCENE_XML}\n'
        'Run the main notebook scene-build cells to generate it.')

print(f'Mitsuba variant : {mi.variant()}')
print(f'Loading scene from {SCENE_XML} ...')
scene = load_scene(SCENE_XML)
scene.frequency = FREQUENCY_HZ

scene.tx_array = PlanarArray(
    num_rows=1, num_cols=1,
    vertical_spacing=0.5, horizontal_spacing=0.5,
    pattern='iso', polarization='V')
scene.rx_array = PlanarArray(
    num_rows=1, num_cols=1,
    vertical_spacing=0.5, horizontal_spacing=0.5,
    pattern='iso', polarization='V')

# ── Material EM setup ────────────────────────────────────────────────────
# CUDA_VISIBLE_DEVICES='' (set in CELL 0) makes DrJit create CPU DLPack
# capsules only, so setting TF-backed properties no longer crashes.
_MAT_PARAMS = {
    # name                    eps_r   sigma    S     xpd   (ITU-R P.2040-2)
    'itu_concrete'          : (5.24,  0.130, 0.40, 0.10),
    'itu_brick'             : (3.91,  0.024, 0.30, 0.10),
    'itu_glass'             : (6.27,  0.012, 0.08, 0.05),  # smooth → low cross-pol
    'itu_wood'              : (1.99,  0.005, 0.25, 0.10),
    'itu_metal'             : (1.00,  1e7,   0.05, 0.01),  # specular → minimal cross-pol
    'itu_asphalt'           : (3.00,  0.010, 0.35, 0.15),
    'itu_vegetation'        : (1.30,  0.001, 0.75, 0.30),  # rough → high cross-pol
    'itu_water'             : (81.0,  0.500, 0.02, 0.05),
    'itu_wet_ground'        : (30.0,  0.150, 0.20, 0.20),
    'itu_medium_dry_ground' : (15.0,  0.035, 0.18, 0.20),
    'itu_very_dry_ground'   : (3.00,  0.001, 0.12, 0.15),
    'itu_marble'            : (7.07,  0.020, 0.08, 0.05),
    'itu_plasterboard'      : (2.73,  0.010, 0.12, 0.10),
    'itu_plywood'           : (2.90,  0.013, 0.20, 0.10),
    'itu_ceiling_board'     : (2.25,  0.006, 0.15, 0.10),
    'itu_chipboard'         : (2.58,  0.011, 0.22, 0.10),
    'itu_floorboard'        : (2.50,  0.008, 0.18, 0.10),
}

try:
    from sionna.rt import LambertianPattern as _LambertianPattern
    _lambertian = _LambertianPattern()
    print('LambertianPattern: loaded')
except ImportError:
    _lambertian = None
    print('LambertianPattern: not available in this Sionna build')

def _setup_mat(name, eps_r, sigma, S, xpd=0.1):
    if name not in scene.radio_materials:
        try: scene.add(RadioMaterial(name))
        except Exception: return
    m = scene.radio_materials[name]
    for attr in ('relative_permittivity', 'permittivity'):
        if hasattr(m, attr):
            try: setattr(m, attr, float(eps_r)); break
            except: pass
    if hasattr(m, 'conductivity'):
        try: m.conductivity = float(sigma)
        except: pass
    for attr in ('scattering_coefficient', 'scattering_coeff'):
        if hasattr(m, attr):
            try: setattr(m, attr, float(S)); break
            except: pass
    if _lambertian is not None:
        for attr in ('scattering_pattern', 'scatter_pattern'):
            if hasattr(m, attr):
                try: setattr(m, attr, _lambertian); break
                except: pass
    # XPD coefficient — cross-polarization discrimination (ITU-R P.2040-2)
    for attr in ('xpd_coefficient', 'cross_polarization_discrimination'):
        if hasattr(m, attr):
            try: setattr(m, attr, float(xpd)); break
            except: pass
    for attr in ('_is_placeholder', 'is_placeholder'):
        if hasattr(m, attr):
            try: object.__setattr__(m, attr, False); break
            except:
                try: setattr(m, attr, False)
                except: pass
            break

for _name, _mat in scene.radio_materials.items():
    if _name == 'vacuum': continue
    _p = _MAT_PARAMS.get(_name)
    if _p:
        _setup_mat(_name, *_p)
    else:
        for attr in ('_is_placeholder',):
            try: object.__setattr__(_mat, attr, False)
            except: pass


_placeholders = [n for n, m in scene.radio_materials.items()
                 if n != 'vacuum' and getattr(m, 'is_placeholder', False)]
if _placeholders:
    print(f'  WARNING placeholder materials: {_placeholders}')
# ── End material setup ─────────────────────────────────────────────────────

print(f'Scene loaded  : {len(scene.objects)} objects  |  {len(scene.radio_materials)} materials')
print(f'Frequency     : {FREQUENCY_HZ/1e9:.3f} GHz')
print()
print(f'  {"Material":<28}  {"eps_r":>6}  {"S":>5}  ph')
print(f'  {"-"*50}')
for _n, _m in scene.radio_materials.items():
    if _n == 'vacuum': continue
    _eps = getattr(_m, 'relative_permittivity', '?')
    _ph  = getattr(_m, 'is_placeholder', '-')
    _S   = getattr(_m, 'scattering_coefficient',
           getattr(_m, 'scattering_coeff', '?'))
    try:    _S_s = f'{float(_S):.2f}'
    except: _S_s = str(_S)
    print(f'  {_n:<28}  {str(_eps)[:6]:>6}  {_S_s:>5}  {_ph}')
print()

# ── scene.preview() ────────────────────────────────────────────────────────
try:
    scene.preview()
except Exception as _e:
    print(f'scene.preview() skipped: {_e}')


## CELL 3 · Scene Preview (pythreejs)

Interactive 3-D scene widget — rotate/zoom to verify geometry and TX position before calibration.

**Optional** — skip if running headless.

In [ ]:
# ====================================================================
# CELL 4b — SCENE PREVIEW  (pythreejs interactive 3D widget)
# ====================================================================
# Renders an interactive 3D scene in the notebook.
# Requires pythreejs: pip install pythreejs
# Only works in a live Jupyter kernel — output won't show in static HTML.
try:
    scene.preview()
except Exception as _pe:
    print(f'scene.preview() failed: {_pe}')


## CELL 4 · Load TX + RX + Measurements

Places transmitter and all 4000 receivers into the scene.

**Reads:**
- `transmitter_positions.csv` → TX position
- `receiver_locations.csv` → 4000 RX (`RX_XXXXXX` IDs, 0–5 km from TX)
- `measurements_with_pathloss.csv` → Ofcom RSSI + path loss

**Requires:** main notebook CELL 6c + CELL 7 to have been run first

In [ ]:
import time as _time

# ── [1/4] Clear previous transmitters ────────────────────────────────────────
print('[1/4] Clearing previous transmitters ...')
for nm in list(scene.transmitters.keys()):
    scene.remove(nm)
print('  Cleared')

# ── [2/4] Load TX CSV or use scene centre ─────────────────────────────────────
print('\n[2/4] Loading transmitter ...')
transmitters = []
_t0 = _time.time()

if os.path.exists(TX_CSV):
    df_tx = pd.read_csv(TX_CSV)
    print(f'  Loaded {len(df_tx)} TX from {TX_CSV}')
    for i, row in df_tx.iterrows():
        lon    = float(row['lon']); lat = float(row['lat'])
        tx_agl = float(row.get('height', TX_AGL_M))
        x, y, _ = gps_to_local(lon, lat)
        ground_z = ray_cast_ground_z(x, y)
        z  = ground_z + tx_agl
        nm = str(row.get('name', f'tx{i:04d}'))
        tx = Transmitter(name=nm, position=(float(x), float(y), float(z)))
        scene.add(tx); transmitters.append(tx)
else:
    # No CSV — use Ofcom-calibrated TX_LON/TX_LAT from CELL 0c
    print(f'  TX CSV not found – using Ofcom site coords TX_LON={TX_LON}, TX_LAT={TX_LAT}')
    x, y, _ = gps_to_local(TX_LON, TX_LAT)
    ground_z = ray_cast_ground_z(x, y)
    z  = ground_z + TX_AGL_M
    tx = Transmitter(name='tx_ofcom', position=(float(x), float(y), float(z)))
    scene.add(tx); transmitters.append(tx)

# Reference TX for downstream cells
tx     = transmitters[0]
abs_z  = _safe(tx.position[2])
tx_agl = TX_AGL_M

# ── [3/4] Antenna arrays ──────────────────────────────────────────────────────
print('\n[3/4] Configuring antenna arrays ...')
scene.tx_array = PlanarArray(
    num_rows=1, num_cols=1,
    vertical_spacing=0.5, horizontal_spacing=0.5,
    pattern='iso', polarization='V')
scene.rx_array = PlanarArray(
    num_rows=1, num_cols=1,
    vertical_spacing=0.5, horizontal_spacing=0.5,
    pattern='iso', polarization='V')
print('  TX: 1x1 isotropic V-pol')
print('  RX: 1x1 isotropic V-pol')

# ── [4/4] TX Summary ─────────────────────────────────────────────────────────
print('\n[4/4] Transmitter summary:')
for t in transmitters:
    x  = _safe(t.position[0])
    y  = _safe(t.position[1])
    z  = _safe(t.position[2])
    lon, lat = local_to_gps(x, y)
    print(f'  TX "{t.name}"  GPS=({lon:.5f},{lat:.5f})  '
          f'XY=({x:.1f},{y:.1f})  Z={z:.2f} m  '
          f'AGL={TX_AGL_M:.1f} m  EIRP={EIRP_DBM:.1f} dBm')
print(f'\n  Done in {_time.time()-_t0:.2f}s')

# ── Load Receivers ────────────────────────────────────────────────────────────
print('\n--- Loading Receivers ---')
import time as _time2

print('[1/5] Loading receiver CSV ...')
if not os.path.exists(RX_CSV):
    print(f'  RX CSV not found at {RX_CSV}')
    df_rx = None
else:
    df_rx = pd.read_csv(RX_CSV)
    print(f'  Loaded {len(df_rx)} receivers')

print('\n[2/5] Clearing previous receivers ...')
for nm in list(scene.receivers.keys()):
    scene.remove(nm)
print(f'  Cleared')

print('\n[3/5] Converting coordinates and computing ground heights ...')
receivers = []
_t0 = _time2.time()

if df_rx is not None:
    for i, row in df_rx.iterrows():
        lon = float(row['lon']); lat = float(row['lat'])
        agl = float(row.get('height', RX_AGL_M))
        x, y, _ = gps_to_local(lon, lat)
        ground_z = ray_cast_ground_z(x, y)
        z = ground_z + agl
        nm = str(row.get('name', f'RX_{i+1:04d}'))
        rx = Receiver(name=nm, position=(float(x), float(y), float(z)))
        rx._ground_z = ground_z
        rx._agl      = agl
        scene.add(rx)
        receivers.append(rx)
    print(f'  Loaded {len(receivers)} receivers in {_time2.time()-_t0:.2f}s')
else:
    rx = Receiver(name='rx0', position=(100.0, 0.0, RX_AGL_M))
    rx._ground_z = 0.0; rx._agl = RX_AGL_M
    scene.add(rx); receivers.append(rx)
    print('  Default single receiver placed')

print('\n[4/5] Validation (first 5 receivers):')
for rx in receivers[:5]:
    x  = _safe(rx.position[0])
    y  = _safe(rx.position[1])
    z  = _safe(rx.position[2])
    gz = getattr(rx, '_ground_z', z - getattr(rx, '_agl', RX_AGL_M))
    lon, lat = local_to_gps(x, y)
    print(f'  {rx.name}: XY({x:.1f}, {y:.1f})  Z={z:.2f}  '
          f'(ground={gz:.2f})  GPS=({lon:.5f},{lat:.5f})')

print('\n[5/5] Bbox containment:')
try:
    _sw = gps_to_local(WEST, SOUTH)
    _ne = gps_to_local(EAST, NORTH)
    xs = [_safe(r.position[0]) for r in receivers]
    ys = [_safe(r.position[1]) for r in receivers]
    x_ok = all(_sw[0] - 50 <= x <= _ne[0] + 50 for x in xs)
    y_ok = all(_sw[1] - 50 <= y <= _ne[1] + 50 for y in ys)
    print(f'  All X inside GPS bounds: {x_ok},  Y inside: {y_ok}')
    if not (x_ok and y_ok):
        print('  WARNING: Some receivers outside GPS bounds — check GPS coordinates')
except Exception as _e:
    print(f'  Bbox check skipped: {_e}')

print(f'\nScene: {len(scene.transmitters)} TX,  {len(scene.receivers)} RX')

# ── Load Ofcom measurements ────────────────────────────────────────────────────
df_meas = None
if os.path.exists(MEASUREMENT_CSV):
    df_meas = pd.read_csv(MEASUREMENT_CSV)
    # Expect columns: lon, lat, rssi_dbm (or pathloss_db)
    if 'rssi_dbm' not in df_meas.columns and 'pathloss_db' in df_meas.columns:
        df_meas['rssi_dbm'] = EIRP_DBM + SYS_GAIN - df_meas['pathloss_db']
    print(f'Measurements: {len(df_meas)} rows   RSSI range [{df_meas["rssi_dbm"].min():.1f}, {df_meas["rssi_dbm"].max():.1f}] dBm')
else:
    print(f'WARNING: measurements CSV not found at {MEASUREMENT_CSV}')


## CELL 5 · Pre-Calibration Baseline

Computes coverage map and RMSE with **default ITU-R P.2040-2 parameters** (before calibration).

**Record this RMSE** — it's your before-calibration benchmark to compare against CELL 8 output.

Expected baseline RMSE: ~12–20 dB

In [ ]:
# ====================================================================
# CELL 4 — PRE-CALIBRATION BASELINE
# ====================================================================
# Quick coverage map with ITU default materials + RMSE vs Ofcom
import time

print('Computing pre-calibration coverage map ...')
_gps_sw = gps_to_local(WEST, SOUTH)
_gps_ne = gps_to_local(EAST, NORTH)
gx_min, gx_max = _gps_sw[0], _gps_ne[0]
gy_min, gy_max = _gps_sw[1], _gps_ne[1]
cx, cy = (gx_min+gx_max)/2, (gy_min+gy_max)/2
cm_z = ray_cast_ground_z(cx, cy) + RX_AGL_M

def _compute_cm_simple(scattering=True, num_samples=NUM_SAMPLES_CM, label=''):
    """Run coverage map in 10M batches and average."""
    SPR = 10_000_000
    n_runs = max(1, num_samples // SPR)
    _is_cpu = 'llvm' in mi.variant() or 'scalar' in mi.variant()
    if _is_cpu:
        n_runs = min(n_runs, 5)
    accum = None
    t0 = time.time()
    for run in range(n_runs):
        cm = scene.coverage_map(
            cm_center=(cx, cy, cm_z), cm_orientation=(0.,0.,0.),
            cm_size=(gx_max-gx_min, gy_max-gy_min),
            cm_cell_size=(GRID_SIZE_M, GRID_SIZE_M),
            max_depth=MAX_DEPTH, num_samples=SPR,
            los=True, reflection=True, scattering=scattering, diffraction=not _is_cpu)
        pg = np.array(cm.path_gain)
        accum = pg if accum is None else accum + pg
        print(f'  {label} run {run+1}/{n_runs}  {time.time()-t0:.0f}s', flush=True)
    avg = accum / n_runs
    pl = -10.0 * np.log10(np.maximum(avg[0] if avg.ndim==3 else avg, 1e-20))
    rssi = EIRP_DBM - pl + SYS_GAIN
    return rssi, pl

rssi_pre, pl_pre = _compute_cm_simple(True, NUM_SAMPLES_CM, 'PRE')
print(f'Pre-cal coverage map done.  RSSI range [{np.nanmin(rssi_pre):.1f}, {np.nanmax(rssi_pre):.1f}] dBm')

# ── RMSE vs Ofcom ─────────────────────────────────────────────────────────────
pre_rmse = pre_r2 = None
if df_meas is not None:
    from scipy.spatial import KDTree as _KDTree
    _xx = np.linspace(gx_min, gx_max, rssi_pre.shape[1])
    _yy = np.linspace(gy_min, gy_max, rssi_pre.shape[0])
    _XX, _YY = np.meshgrid(_xx, _yy)
    grid_pts = np.column_stack([_XX.ravel(), _YY.ravel()])
    grid_rssi = rssi_pre.ravel()
    valid = np.isfinite(grid_rssi) & (grid_rssi > NOISE_FLOOR)
    _kd = _KDTree(grid_pts[valid])
    rx_xy = np.array([gps_to_local(r['lon'], r['lat'])[:2] for _, r in df_meas.iterrows()])
    _, idx = _kd.query(rx_xy)
    pred = grid_rssi[valid][idx]
    meas = df_meas['rssi_dbm'].values
    err = pred - meas
    pre_rmse = float(np.sqrt(np.mean(err**2)))
    ss_res = np.sum(err**2)
    ss_tot = np.sum((meas - meas.mean())**2)
    pre_r2 = float(1.0 - ss_res/ss_tot)
    print(f'Pre-cal RMSE={pre_rmse:.2f} dB   R\u00b2={pre_r2:.4f}')

# ── Plot ───────────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(rssi_pre, origin='lower', extent=[gx_min,gx_max,gy_min,gy_max],
               cmap='jet', aspect='auto', vmin=-120, vmax=-40)
_tx = list(scene.transmitters.values())[0]
ax.scatter(_safe(_tx.position[0]), _safe(_tx.position[1]), marker='*', s=400,
           c='gold', edgecolors='black', linewidths=0.8, zorder=10, label='TX')
plt.colorbar(im, ax=ax, label='RSSI (dBm)')
ax.set_title(f'Pre-calibration RSSI (ITU defaults)'
             + (f'  RMSE={pre_rmse:.1f} dB  R\u00b2={pre_r2:.3f}' if pre_rmse else ''))
ax.axis('off')
ax.legend(loc='upper right')
plt.tight_layout()
_out = os.path.join(OUT_DIR, 'pre_cal_coverage.png')
plt.savefig(_out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved \u2192 {_out}')


## CELL 6 · Calibration Data Preparation

Selects a balanced stratified sample of 200 Ofcom drive-test RX points for calibration.

- Stratified across 500 m distance bins (avoids near-TX bias)
- Adds `cal_XXXX` receivers to the scene
- Prepares `rssi_measured` as `tf.constant` — the ground-truth calibration target

**Following:** diff-rt-calibration (Hoydis et al., NeurIPS 2023) — measured power as target

In [ ]:
# ====================================================================
# CELL 6 — CALIBRATION DATA PREPARATION
# ====================================================================
# Selects a balanced subset of measured RX points for calibration.
# Following diff-rt-calibration (Hoydis et al. 2023): calibration target
# is MEASURED signal power, not a simulated reference.
# Uses compute_paths() — the only differentiable path to material gradients.
# coverage_map() cannot be used: the grid NN-lookup breaks the gradient chain.
# ====================================================================
import tensorflow as tf
import numpy as np

print('CELL 6 — Calibration data preparation')
print('=' * 60)

if df_meas is None:
    raise RuntimeError('df_meas not loaded — run CELL 4 first')

# ── Select calibration subset — balanced across distance bins ─────────────────
CALIB_N_RX   = 200    # number of RX to use for calibration (speed vs accuracy)
CALIB_BATCH  = 50     # RX per gradient step (mini-batch, like diff-rt-calibration)

_df = df_meas.copy().reset_index(drop=True)

# Compute distance from TX for each measurement
_tx_x, _tx_y = _safe(list(scene.transmitters.values())[0].position[0]), \
               _safe(list(scene.transmitters.values())[0].position[1])
_rx_xy = np.array([gps_to_local(r['lon'], r['lat'])[:2] for _, r in _df.iterrows()])
_df['dist_m'] = np.sqrt((_rx_xy[:,0] - _tx_x)**2 + (_rx_xy[:,1] - _tx_y)**2)
_df['rx_x']   = _rx_xy[:,0]
_df['rx_y']   = _rx_xy[:,1]

# Stratified sampling: equal samples from each 500m distance bin
_bins = np.arange(0, _df['dist_m'].max() + 500, 500)
_df['dist_bin'] = pd.cut(_df['dist_m'], bins=_bins, labels=False)
_per_bin = max(1, CALIB_N_RX // len(_bins))
_frames  = []
for _b in _df['dist_bin'].dropna().unique():
    _sub = _df[_df['dist_bin'] == _b]
    _frames.append(_sub.sample(min(len(_sub), _per_bin), random_state=42))
df_calib = pd.concat(_frames).sample(frac=1, random_state=42).reset_index(drop=True)
df_calib  = df_calib.head(CALIB_N_RX)

print(f'Calibration RX   : {len(df_calib)} / {len(df_meas)} total')
print(f'Distance range   : {df_calib["dist_m"].min():.0f} – {df_calib["dist_m"].max():.0f} m')
print(f'RSSI range       : {df_calib["rssi_dbm"].min():.1f} – {df_calib["rssi_dbm"].max():.1f} dBm')
print(f'Mini-batch size  : {CALIB_BATCH}  ({len(df_calib)//CALIB_BATCH} batches/epoch)')

# ── Add calibration RX to scene ───────────────────────────────────────────────
print('\nAdding calibration receivers to scene ...')
for nm in list(scene.receivers.keys()):
    if nm.startswith('cal_'): scene.remove(nm)

calib_receivers = []
for i, row in df_calib.iterrows():
    x, y = float(row['rx_x']), float(row['rx_y'])
    gz   = ray_cast_ground_z(x, y)
    z    = gz + RX_AGL_M
    nm   = f'cal_{i:04d}'
    rx   = Receiver(name=nm, position=(x, y, z))
    scene.add(rx)
    calib_receivers.append(rx)

# Ground-truth RSSI as tf.constant (the calibration target)
rssi_measured = tf.constant(df_calib['rssi_dbm'].values, dtype=tf.float32)

print(f'Added {len(calib_receivers)} cal_ receivers to scene')
print(f'Measured RSSI tensor: {rssi_measured.shape}  mean={float(tf.reduce_mean(rssi_measured)):.1f} dBm')
print('\nCalibration data ready.')


## CELL 7 · Differentiable RT Material Calibration

Optimises `eps_r`, `sigma`, `S` for each ITU material using **Adam + power loss**.

| Item | Value |
|---|---|
| Loss | `MSE(sim_RSSI_dBm, measured_RSSI_dBm)` — power loss, dB domain |
| Method | `compute_paths()` per mini-batch — differentiable, exact per-RX power |
| Target | Measured Ofcom RSSI (`rssi_measured`) — **not** a self-simulated reference |
| Batch | 50 receivers per gradient step |
| Steps | 100 Adam steps |

**Following:** diff-rt-calibration (Hoydis et al.): `power_loss` on per-receiver received power vs measurements.

**Output:** `calibrated_materials.json` — copy values to main notebook CELL 5 `_ITU_DB` to improve RMSE.

In [ ]:
# ====================================================================
# CELL 7 — DIFFERENTIABLE RT MATERIAL CALIBRATION
# ====================================================================
# Minimises MSE between simulated RSSI and measured Ofcom RSSI.
# Following diff-rt-calibration (Hoydis et al. 2023):
#   - compute_paths() for exact per-RX power (differentiable)
#   - power_loss = MSE(sim_RSSI_dBm, measured_RSSI_dBm)
#   - Mini-batch gradient steps (CALIB_BATCH RX per step)
#   - Adam optimiser on eps_r, sigma, S per material
# ====================================================================
import tensorflow as tf
import numpy as np

print('CELL 7 — Differentiable RT Material Calibration')
print('=' * 60)

_is_cpu = 'llvm' in mi.variant() or 'scalar' in mi.variant()
_tx_pwr_w = 10.0**((EIRP_DBM - 30.0) / 10.0)   # EIRP in Watts

# ── Make material EM properties trainable tf.Variables ───────────────────────
# Only optimise materials that appear in the scene geometry.
_CALIB_MATS = [
    'itu_concrete', 'itu_brick', 'itu_glass', 'itu_wood',
    'itu_metal', 'itu_vegetation', 'itu_medium_dry_ground',
    'itu_water', 'itu_very_dry_ground',
]

train_vars  = []   # list of tf.Variable
var_meta    = []   # (mat_name, attr_name, variable)
orig_params = {}

for _mname in _CALIB_MATS:
    if _mname not in scene.radio_materials:
        continue
    _mat = scene.radio_materials[_mname]
    orig_params[_mname] = {}
    for _attr, _lo, _hi in [
        ('relative_permittivity', 1.0,  80.0),
        ('conductivity',          1e-4,  1e7),
        ('scattering_coefficient',0.01,  0.99),
    ]:
        _val_raw = getattr(_mat, _attr, None)
        if _val_raw is None:
            continue
        _val = float(_safe(_val_raw))
        orig_params[_mname][_attr] = _val
        # Create a tf.Variable and assign it as the material property
        _var = tf.Variable(_val, trainable=True, dtype=tf.float32,
                           name=f'{_mname}_{_attr}')
        try:
            if hasattr(_val_raw, 'assign'):
                _val_raw.assign(_var)
            else:
                setattr(_mat, _attr, _var)
        except Exception:
            pass
        train_vars.append(_var)
        var_meta.append((_mname, _attr, _var, _lo, _hi))

print(f'Trainable variables : {len(train_vars)}')
print(f'Materials           : {list(orig_params.keys())}')

# ── Power loss: MSE of RSSI in dB ────────────────────────────────────────────
def compute_rssi_batch(rx_indices):
    """
    Run compute_paths for a subset of cal receivers.
    Returns simulated RSSI [dBm] as tf.Tensor shape [len(rx_indices)].
    Following diff-rt-calibration: power_loss on per-RX received power.
    """
    # Temporarily set only batch receivers as active — Sionna 0.19 uses
    # all receivers in scene.receivers, so we use the full set and index.
    paths = scene.compute_paths(
        max_depth          = CALIB_DEPTH,
        num_samples        = CALIB_NUM_SAMP,
        los                = True,
        specular_reflection= True,
        diffuse_reflection = True,
        refraction         = False,
        diffraction        = False,
    )
    try:
        a, _ = paths.cir()
        # a shape: [batch, num_tx, num_rx, max_paths, num_time_steps]
        # squeeze TX dim (1 TX), sum over paths/time → per-RX power
        a = tf.squeeze(a)                       # [num_rx, max_paths, T] or similar
        # Handle varying shape
        while len(a.shape) > 2:
            a = tf.reduce_sum(tf.abs(a)**2, axis=-1)
        if len(a.shape) == 2:
            pwr = tf.reduce_sum(tf.abs(a)**2, axis=-1)   # [num_rx]
        else:
            pwr = tf.abs(a)**2
        pwr = tf.cast(pwr, tf.float32)
        # Convert path gain to RSSI
        rssi = 10.0 * tf.math.log(pwr * _tx_pwr_w + 1e-30) / tf.math.log(10.0) + 30.0 + SYS_GAIN
        return tf.gather(rssi, rx_indices)
    except Exception as _e:
        print(f'  cir() error: {_e}')
        return tf.zeros(len(rx_indices), dtype=tf.float32) + NOISE_FLOOR

def power_loss(rssi_sim, rssi_meas):
    """MSE in dB — equivalent to power_loss in diff-rt-calibration."""
    err = rssi_sim - tf.cast(rssi_meas, rssi_sim.dtype)
    return tf.reduce_mean(err ** 2)

def clamp_vars():
    for _, _, var, lo, hi in var_meta:
        var.assign(tf.clip_by_value(var, lo, hi))

# ── Training loop — mini-batch gradient descent ───────────────────────────────
optimizer = tf.keras.optimizers.Adam(learning_rate=CALIB_LR)
N_RX      = len(calib_receivers)
indices   = np.arange(N_RX)
history   = {'step': [], 'rmse_db': [], 'loss': []}

print(f'\nStarting calibration: {CALIB_STEPS} steps  LR={CALIB_LR}')
print(f'  Batch={CALIB_BATCH}  depth={CALIB_DEPTH}  samples={CALIB_NUM_SAMP:,}')
print(f'  Target: measured Ofcom RSSI ({N_RX} receivers)')
print('-' * 60)

t0 = time.time()
for step in range(CALIB_STEPS):
    # Mini-batch: random subset of calibration receivers
    np.random.shuffle(indices)
    batch_idx = indices[:CALIB_BATCH].tolist()
    rssi_meas_batch = tf.gather(rssi_measured, batch_idx)

    with tf.GradientTape() as tape:
        tape.watch(train_vars)
        rssi_sim_batch = compute_rssi_batch(batch_idx)
        loss = power_loss(rssi_sim_batch, rssi_meas_batch)

    grads = tape.gradient(loss, train_vars,
                          unconnected_gradients=tf.UnconnectedGradients.ZERO)
    optimizer.apply_gradients(zip(grads, train_vars))
    clamp_vars()

    rmse = float(tf.sqrt(loss).numpy())
    history['step'].append(step)
    history['rmse_db'].append(rmse)
    history['loss'].append(float(loss.numpy()))

    if step % 10 == 0 or step == CALIB_STEPS - 1:
        n_zero = sum(1 for g in grads
                     if g is None or float(tf.reduce_sum(tf.abs(g)).numpy()) < 1e-12)
        print(f'  step {step:4d}/{CALIB_STEPS}  RMSE={rmse:.2f} dB  '
              f'zero_grads={n_zero}/{len(train_vars)}  t={time.time()-t0:.0f}s',
              flush=True)

print(f'\nCalibration done in {time.time()-t0:.1f}s')
print(f'Final RMSE: {history["rmse_db"][-1]:.2f} dB  '
      f'(initial: {history["rmse_db"][0]:.2f} dB)')

# ── Plot convergence ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(history['step'], history['rmse_db'], 'b-', lw=2)
ax.set_xlabel('Step'); ax.set_ylabel('RMSE (dB)')
ax.set_title('Calibration convergence — power loss vs measured Ofcom RSSI')
ax.grid(True, alpha=0.4)
plt.tight_layout()
_out = os.path.join(OUT_DIR, 'calibration_loss.png')
plt.savefig(_out, dpi=150, bbox_inches='tight'); plt.show()
print(f'Saved → {_out}')

# ── Print calibrated values ───────────────────────────────────────────────────
print(f'\n{"Material":<30} {"Parameter":<26} {"Initial":>10} {"Calibrated":>12} {"Delta":>8}')
print('=' * 90)
calib_results = {}
for _mname, _attr, _var, _, _ in var_meta:
    _init = orig_params[_mname].get(_attr, float('nan'))
    _cal  = float(_var.numpy())
    _delta = _cal - _init
    print(f'  {_mname:<28}  {_attr:<24}  {_init:>10.4f}  {_cal:>12.4f}  {_delta:>+8.4f}')
    if _mname not in calib_results:
        calib_results[_mname] = {}
    _pname = {'relative_permittivity': 'eps_r',
              'conductivity': 'sigma',
              'scattering_coefficient': 'S'}.get(_attr, _attr)
    calib_results[_mname][_pname] = _cal
print()

import json as _json
_cal_json = os.path.join(OUT_DIR, 'calibrated_materials.json')
with open(_cal_json, 'w') as _f:
    _json.dump(calib_results, _f, indent=2)
print(f'Calibrated materials → {_cal_json}')
print('\nNext: run CELL 8 (TX orientation) then CELL 9 (post-cal analysis)')


## CELL 8 · TX Orientation Optimisation

Optimises TX antenna orientation (azimuth/tilt) using **RMSprop** to minimise NMSE.

| Parameter | Value |
|---|---|
| `ORI_STEPS` | 50 |
| `ORI_LR` | 1e-2 |

**Output:** optimised `[alpha, beta, gamma]` orientation — update TX in main notebook CELL 6 if improved.

In [ ]:
# ====================================================================
# CELL 7 — TX ORIENTATION OPTIMISATION  (optional)
# ====================================================================
tx_name = list(scene.transmitters.keys())[0]
tx_obj  = scene.transmitters[tx_name]

# Make orientation a trainable tf.Variable
_ori_init = [_safe(tx_obj.orientation[i]) for i in range(3)]
tx_orientation = tf.Variable(_ori_init, trainable=True, dtype=tf.float32)

def set_tx_orientation(ori_var):
    try: tx_obj.orientation = ori_var
    except Exception:
        pass  # some Sionna builds don't support direct assignment

ori_optimizer = tf.keras.optimizers.RMSprop(learning_rate=ORI_LR)
ori_history   = {'step':[], 'nmse_db':[]}

print(f'TX orientation optimisation: {ORI_STEPS} steps  LR={ORI_LR}')
print(f'  Initial orientation: {_ori_init}')

t0 = time.time()
for step in range(ORI_STEPS):
    with tf.GradientTape() as tape:
        tape.watch(tx_orientation)
        set_tx_orientation(tx_orientation)
        h_hat = compute_h_freq(scene, num_samp=CALIB_NUM_SAMP, depth=CALIB_DEPTH)
        loss  = nmse_loss(h_hat, h_ref_tf)
        nmse_db = 10.0 * tf.math.log(loss + 1e-12) / tf.math.log(10.0)
    grads = tape.gradient(loss, [tx_orientation])
    ori_optimizer.apply_gradients(zip(grads, [tx_orientation]))
    # clamp azimuth/elevation to +-pi
    tx_orientation.assign(tf.clip_by_value(tx_orientation, -3.14159, 3.14159))
    ori_history['step'].append(step)
    ori_history['nmse_db'].append(float(nmse_db))
    if step % 10 == 0 or step == ORI_STEPS-1:
        print(f'  step {step:3d}/{ORI_STEPS}  NMSE={float(nmse_db):.2f} dB  '
              f'ori={[round(float(tx_orientation[i]),3) for i in range(3)]}  '
              f'({time.time()-t0:.0f}s)', flush=True)

print(f'TX orientation optimised in {time.time()-t0:.1f}s')
_cal_ori = [float(tx_orientation[i]) for i in range(3)]
print(f'Final orientation (rad): {_cal_ori}')


## CELL 9 · Post-Calibration Analysis

Recomputes coverage map with calibrated parameters and compares against:
- Pre-calibration baseline (CELL 5)
- Ofcom drive-test measurements

**Expected improvement:** 3–8 dB RMSE reduction

**Output files:**
- `results_diff_rt/post_cal_metrics.csv`
- `results_diff_rt/calibration_summary.png`

**Next step:** copy calibrated `_MAT_PARAMS` to main notebook CELL 3, re-run CELL 8 → 9b → 9d/9e for final RMSE.

In [ ]:
# ====================================================================
# CELL 8 — POST-CALIBRATION ANALYSIS
# ====================================================================
print('Computing post-calibration coverage map ...')
rssi_post, pl_post = _compute_cm_simple(True, NUM_SAMPLES_CM, 'POST')
print(f'Post-cal RSSI range [{np.nanmin(rssi_post):.1f}, {np.nanmax(rssi_post):.1f}] dBm')

# ── Metrics vs Ofcom ──────────────────────────────────────────────────────────
def _metrics_vs_meas(rssi_grid, label):
    if df_meas is None:
        print(f'{label}: no measurements'); return {}
    _xx = np.linspace(gx_min, gx_max, rssi_grid.shape[1])
    _yy = np.linspace(gy_min, gy_max, rssi_grid.shape[0])
    _XX, _YY = np.meshgrid(_xx, _yy)
    grid_pts  = np.column_stack([_XX.ravel(), _YY.ravel()])
    grid_rssi = rssi_grid.ravel()
    valid = np.isfinite(grid_rssi) & (grid_rssi > NOISE_FLOOR)
    from scipy.spatial import KDTree as _KD
    _kd = _KD(grid_pts[valid])
    rx_xy = np.array([gps_to_local(r['lon'], r['lat'])[:2] for _, r in df_meas.iterrows()])
    _, idx = _kd.query(rx_xy)
    pred = grid_rssi[valid][idx]
    meas = df_meas['rssi_dbm'].values
    err  = pred - meas
    mse  = float(np.mean(err**2))
    rmse = float(np.sqrt(mse))
    mae  = float(np.mean(np.abs(err)))
    bias = float(np.mean(err))
    ss_res = float(np.sum(err**2))
    ss_tot = float(np.sum((meas - meas.mean())**2))
    r2   = 1.0 - ss_res / ss_tot
    print(f'{label:20s}: n={len(meas)}  RMSE={rmse:.2f} dB  MAE={mae:.2f}  Bias={bias:+.2f}  R\u00b2={r2:.4f}')
    return {'label':label,'n':len(meas),'mse':mse,'rmse':rmse,'mae':mae,'bias':bias,'r2':r2}

metrics_pre  = _metrics_vs_meas(rssi_pre,  'Pre-calibration')
metrics_post = _metrics_vs_meas(rssi_post, 'Post-calibration')

# ── Save metrics CSV ──────────────────────────────────────────────────────────
import pandas as pd
_rows = [r for r in [metrics_pre, metrics_post] if r]
if _rows:
    pd.DataFrame(_rows).to_csv(os.path.join(OUT_DIR, 'calibration_metrics.csv'), index=False)
    print(f'Metrics saved \u2192 {os.path.join(OUT_DIR, "calibration_metrics.csv")}')

# ── Side-by-side plot: pre vs post ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
for ax, data, m, ttl in [
    (axes[0], rssi_pre,  metrics_pre,  'Pre-calibration (ITU defaults)'),
    (axes[1], rssi_post, metrics_post, 'Post-calibration (optimised)'),
]:
    im = ax.imshow(data, origin='lower', extent=[gx_min,gx_max,gy_min,gy_max],
                   cmap='jet', aspect='auto', vmin=-120, vmax=-40)
    _tx_obj = list(scene.transmitters.values())[0]
    ax.scatter(_safe(_tx_obj.position[0]), _safe(_tx_obj.position[1]),
               marker='*', s=400, c='gold', edgecolors='black', linewidths=0.8, zorder=10)
    plt.colorbar(im, ax=ax, label='RSSI (dBm)')
    _sub = (f'  RMSE={m["rmse"]:.1f} dB  R\u00b2={m["r2"]:.3f}' if m else '')
    ax.set_title(ttl + _sub, fontsize=12)
    ax.axis('off')

plt.suptitle(f'Differentiable RT Calibration \u2014 {CITY_NAME}  {FREQUENCY_HZ/1e9:.3f} GHz', fontsize=14)
plt.tight_layout()
_out = os.path.join(OUT_DIR, 'pre_vs_post_calibration.png')
plt.savefig(_out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved \u2192 {_out}')

# ── RSSI improvement histogram ────────────────────────────────────────────────
if df_meas is not None and metrics_pre and metrics_post:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    # Error distribution comparison
    for ax, rssi_g, m, col, lbl in [
        (axes[0], rssi_pre,  metrics_pre,  'steelblue', 'Pre-cal'),
        (axes[0], rssi_post, metrics_post, 'tomato',    'Post-cal'),
    ]:
        _xx = np.linspace(gx_min, gx_max, rssi_g.shape[1])
        _yy = np.linspace(gy_min, gy_max, rssi_g.shape[0])
        _XX, _YY = np.meshgrid(_xx, _yy)
        _gp = np.column_stack([_XX.ravel(), _YY.ravel()])
        _gr = rssi_g.ravel()
        _vld = np.isfinite(_gr) & (_gr > NOISE_FLOOR)
        from scipy.spatial import KDTree as _KD2
        _, _idx = _KD2(_gp[_vld]).query(
            np.array([gps_to_local(r['lon'],r['lat'])[:2] for _,r in df_meas.iterrows()]))
        _pred = _gr[_vld][_idx]
        _err  = _pred - df_meas['rssi_dbm'].values
        ax.hist(_err, bins=40, alpha=0.55, color=col, label=f'{lbl} (RMSE={m["rmse"]:.1f}dB)')
    axes[0].axvline(0, color='k', lw=1, ls='--')
    axes[0].set_xlabel('Prediction error (dB)'); axes[0].set_ylabel('Count')
    axes[0].set_title('Error distribution: pre vs post calibration')
    axes[0].legend()
    # Calibration convergence
    axes[1].plot(history['step'], history['nmse_db'], 'b-', lw=2)
    axes[1].set_xlabel('Step'); axes[1].set_ylabel('NMSE (dB)')
    axes[1].set_title('Calibration convergence (NMSE loss)')
    axes[1].grid(True, alpha=0.4)
    plt.tight_layout()
    _out2 = os.path.join(OUT_DIR, 'calibration_analysis.png')
    plt.savefig(_out2, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved \u2192 {_out2}')

print('\n=== CALIBRATION COMPLETE ===')
print(f'Outputs in: {OUT_DIR}')
